In [ ]:
import os
os.chdir(os.path.join('..', 'results'))

# Strategy 2 — 30 Repeated Runs (PartB: KMeans/Fairlet/BFKM, 8 configs)
**Seeds 0–29. Majority vote k. Runtime recorded.**

In [2]:
## Cell 1: Imports

In [3]:
import numpy as np
import pandas as pd
import pickle
import requests
import io
from scipy.io import arff
from sklearn.cluster import KMeans
# from sklearn.preprocessing import StandardScaler, LabelEncoder  # ORIGINAL (Reviewer 3, Comment 8)
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder  # FIXED: added OneHotEncoder (LabelEncoder import kept, unused elsewhere)
from sklearn.metrics import (
    silhouette_score, silhouette_samples, calinski_harabasz_score,
    davies_bouldin_score, adjusted_rand_score, normalized_mutual_info_score,
    adjusted_mutual_info_score, fowlkes_mallows_score, v_measure_score
)
from scipy.spatial.distance import cdist, pdist
from scipy.stats import entropy
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')
np.random.seed(0)

In [4]:
## Cell 2: Original 5 Dataset Loaders (D1–D5)

In [5]:
# =============================================================================
# SECTION 1: ORIGINAL 5 DATASET LOADERS (unchanged)
# =============================================================================

def load_adult():
    print("\n" + "="*80)
    print("Loading Adult Income Dataset")
    print("="*80)
    columns = [
        'age', 'workclass', 'fnlwgt', 'education', 'education-num',
        'marital-status', 'occupation', 'relationship', 'race', 'sex',
        'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
    ]
    adult = pd.read_csv("../data/adult/adult.data", names=columns, header=None)
    adult = adult.replace(' ?', np.nan).dropna()
    adult['sex_binary'] = (adult['sex'].str.strip() == 'Male').astype(int)
    adult_features = [c for c in adult.columns if c not in ["income", "sex", "race", "sex_binary"]]
    print(f"Shape: {adult.shape}")
    print(f"Gender: 0(F)={( adult['sex_binary']==0).sum()}, 1(M)={(adult['sex_binary']==1).sum()}")
    return adult, adult_features, 'sex_binary'


def load_compas():
    print("\n" + "="*80)
    print("Loading COMPAS Dataset")
    print("="*80)
    compas = pd.read_csv("../data/compas/compas-scores.csv")
    compas = compas[
        (compas['days_b_screening_arrest'] <= 30) &
        (compas['days_b_screening_arrest'] >= -30) &
        (compas['is_recid'] != -1)
    ]
    selected_cols = compas.columns[compas.isnull().sum() == 0].tolist()
    compas = compas[selected_cols]
    compas['sex_binary'] = (compas['sex'] == 'Male').astype(int)
    compas_features = [c for c in compas.columns if c not in ["is_recid", "sex", "race", "sex_binary"]]
    print(f"Shape: {compas.shape}")
    print(f"Gender: 0(F)={(compas['sex_binary']==0).sum()}, 1(M)={(compas['sex_binary']==1).sum()}")
    return compas, compas_features, 'sex_binary'


def load_german():
    print("\n" + "="*80)
    print("Loading German Credit Dataset")
    print("="*80)
    german_columns = [
        'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
        'savings_status', 'employment', 'installment_rate', 'personal_status_sex',
        'other_parties', 'residence_since', 'property_magnitude', 'age',
        'other_payment_plans', 'housing', 'existing_credits', 'job',
        'num_dependents', 'own_telephone', 'foreign_worker', 'class'
    ]
    german = pd.read_csv("../data/german/german.data", sep=' ', names=german_columns, header=None)
    def extract_sex(s):
        return 'male' if s in ['A91', 'A93', 'A94'] else ('female' if s in ['A92', 'A95'] else 'unknown')
    german['sex'] = german['personal_status_sex'].apply(extract_sex)
    german = german[german['sex'] != 'unknown']
    german['sex_binary'] = (german['sex'] == 'male').astype(int)
    german_features = [c for c in german.columns if c not in ["class", "sex", "sex_binary"]]
    print(f"Shape: {german.shape}")
    print(f"Gender: 0(F)={(german['sex_binary']==0).sum()}, 1(M)={(german['sex_binary']==1).sum()}")
    return german, german_features, 'sex_binary'


def load_credit():
    print("\n" + "="*80)
    print("Loading Default Credit Card Dataset")
    print("="*80)
    credit = pd.read_csv("../data/default of credit card clients/default of credit card clients.csv", header=0)
    if 'X1' in credit.columns or credit.columns[0] == 'ID':
        credit.columns = [
            'ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
            'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
            'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
            'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default'
        ]
    credit = credit.drop(columns=['ID'])
    credit['SEX'] = pd.to_numeric(credit['SEX'], errors='coerce')
    credit['sex'] = credit['SEX'].map({1.0: 'male', 2.0: 'female'})
    credit = credit.dropna(subset=['sex'])
    credit['sex_binary'] = (credit['sex'] == 'male').astype(int)
    credit_features = [c for c in credit.columns if c not in ["default", "sex", "SEX", "sex_binary"]]
    print(f"Shape: {credit.shape}")
    print(f"Gender: 0(F)={(credit['sex_binary']==0).sum()}, 1(M)={(credit['sex_binary']==1).sum()}")
    return credit, credit_features, 'sex_binary'


def load_law():
    print("\n" + "="*80)
    print("Loading Law School (LSAC) Dataset")
    print("="*80)
    lsac = pd.read_csv("../data/law/law_dataset.csv")
    lsac['sex'] = lsac['male'].map({1: 'Male', 0: 'Female', 1.0: 'Male', 0.0: 'Female'})
    lsac = lsac[lsac['sex'].notna()]
    lsac['sex_binary'] = (lsac['sex'] == 'Male').astype(int)
    drop_cols = ['male', 'sex', 'pass_bar', 'sex_binary', 'racetxt'] 
    lsac_features = [c for c in lsac.columns if c not in drop_cols]
    print(f"Shape: {lsac.shape}")
    print(f"Gender: 0(F)={(lsac['sex_binary']==0).sum()}, 1(M)={(lsac['sex_binary']==1).sum()}")
    return lsac, lsac_features, 'sex_binary'

# =============================================================================
# RACE & COMBINED LOADERS — Adult, COMPAS, Law
# German, Credit: gender only
# =============================================================================

def load_adult_race():
    columns = [
        'age', 'workclass', 'fnlwgt', 'education', 'education-num',
        'marital-status', 'occupation', 'relationship', 'race', 'sex',
        'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
    ]
    df = pd.read_csv("../data/adult/adult.data", names=columns, header=None)
    df = df.replace(' ?', np.nan).dropna()
    df['race'] = df['race'].str.strip()
    df['race_binary'] = (df['race'] == 'White').astype(int)
    features = [c for c in df.columns if c not in ["income", "sex", "race", "race_binary"]]
    print(f"Adult (race) | 0(NonWhite)={(df['race_binary']==0).sum()}, 1(White)={(df['race_binary']==1).sum()}")
    return df, features, 'race_binary'

def load_adult_combined():
    columns = [
        'age', 'workclass', 'fnlwgt', 'education', 'education-num',
        'marital-status', 'occupation', 'relationship', 'race', 'sex',
        'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
    ]
    df = pd.read_csv("../data/adult/adult.data", names=columns, header=None)
    df = df.replace(' ?', np.nan).dropna()
    df['race'] = df['race'].str.strip()
    df['sex_binary']  = (df['sex'].str.strip() == 'Male').astype(int)
    df['race_binary'] = (df['race'] == 'White').astype(int)
    df['combined']    = df['race_binary'] * 2 + df['sex_binary']
    features = [c for c in df.columns if c not in ["income","sex","race","sex_binary","race_binary","combined"]]
    print(f"Adult (combined):\n{df['combined'].value_counts().sort_index()}")
    print("0=NonWhite-Female, 1=NonWhite-Male, 2=White-Female, 3=White-Male")
    return df, features, 'combined'

def load_compas_race():
    df = pd.read_csv("../data/compas/compas-scores.csv")
    df = df[(df['days_b_screening_arrest']<=30)&(df['days_b_screening_arrest']>=-30)&(df['is_recid']!=-1)]
    df = df[df.columns[df.isnull().sum()==0].tolist()]
    df = df[df['race'].isin(['Caucasian','African-American'])].copy()
    df['race_binary'] = (df['race'] == 'African-American').astype(int)
    features = [c for c in df.columns if c not in ["is_recid","sex","race","race_binary"]]
    print(f"COMPAS (race) | 0(White)={(df['race_binary']==0).sum()}, 1(Black)={(df['race_binary']==1).sum()}")
    return df, features, 'race_binary'

def load_compas_combined():
    df = pd.read_csv("../data/compas/compas-scores.csv")
    df = df[(df['days_b_screening_arrest']<=30)&(df['days_b_screening_arrest']>=-30)&(df['is_recid']!=-1)]
    df = df[df.columns[df.isnull().sum()==0].tolist()]
    df = df[df['race'].isin(['Caucasian','African-American'])].copy()
    df['sex_binary']  = (df['sex'] == 'Male').astype(int)
    df['race_binary'] = (df['race'] == 'African-American').astype(int)
    df['combined']    = df['race_binary'] * 2 + df['sex_binary']
    features = [c for c in df.columns if c not in ["is_recid","sex","race","sex_binary","race_binary","combined"]]
    print(f"COMPAS (combined):\n{df['combined'].value_counts().sort_index()}")
    print("0=White-Female, 1=White-Male, 2=Black-Female, 3=Black-Male")
    return df, features, 'combined'

def load_law_race():
    df = pd.read_csv("../data/law/law_dataset.csv")
    df['sex'] = df['male'].map({1:'Male',0:'Female',1.0:'Male',0.0:'Female'})
    df = df[df['sex'].notna()]
    df['race_binary'] = df['racetxt'].astype(int)  # 1=White, 0=NonWhite
    features = [c for c in df.columns if c not in ['male','sex','pass_bar','racetxt','race_binary']]
    print(f"Law (race) | 0(NonWhite)={(df['race_binary']==0).sum()}, 1(White)={(df['race_binary']==1).sum()}")
    return df, features, 'race_binary'

def load_law_combined():
    df = pd.read_csv("../data/law/law_dataset.csv")
    df['sex'] = df['male'].map({1:'Male',0:'Female',1.0:'Male',0.0:'Female'})
    df = df[df['sex'].notna()]
    df['sex_binary']  = (df['sex'] == 'Male').astype(int)
    df['race_binary'] = df['racetxt'].astype(int)
    df['combined']    = df['race_binary'] * 2 + df['sex_binary']
    features = [c for c in df.columns if c not in ['male','sex','pass_bar','racetxt','sex_binary','race_binary','combined']]
    print(f"Law (combined):\n{df['combined'].value_counts().sort_index()}")
    print("0=NonWhite-Female, 1=NonWhite-Male, 2=White-Female, 3=White-Male")
    return df, features, 'combined'

In [6]:
## Cell 3: New Dataset Loaders (D6–D8)

In [7]:
import os
# SECTION 2: NEW DATASET LOADERS — D6, D7, D8
# Each supports mode='gender' | 'race' | 'combined'
# =============================================================================

def load_diabetes(mode='gender'):
    """
    D6: Diabetes 130-US Hospitals (UCI id=296)
    mode: 'gender' | 'race' | 'combined'
    race: 1=Caucasian, 0=AfricanAmerican
    combined: 0=Black-Female, 1=Black-Male, 2=White-Female, 3=White-Male
    """
    print("\n" + "="*80)
    print(f"Loading Diabetes 130-US Dataset (mode={mode})")
    print("="*80)
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=296)
    df = ds.data.features.copy()

    drop_cols = ['encounter_id', 'patient_nbr', 'weight', 'payer_code', 'medical_specialty']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df = df[df['gender'].isin(['Male', 'Female'])]
    df = df[df['race'] != '?'].dropna(subset=['gender', 'race'])

    df['sex_binary'] = (df['gender'] == 'Male').astype(int)

    # For race/combined: keep only Caucasian and AfricanAmerican
    df_race = df[df['race'].isin(['Caucasian', 'AfricanAmerican'])].copy()
    df_race['race_binary'] = (df_race['race'] == 'Caucasian').astype(int)
    df_race['combined'] = df_race['race_binary'] * 2 + df_race['sex_binary']

    base_excl = ['gender', 'race', 'sex_binary', 'race_binary', 'combined']

    if mode == 'gender':
        features = [c for c in df.columns if c not in base_excl]
        print(f"Shape: {df.shape} | Gender: 0(F)={(df['sex_binary']==0).sum()}, 1(M)={(df['sex_binary']==1).sum()}")
        return df, features, 'sex_binary'
    elif mode == 'race':
        features = [c for c in df_race.columns if c not in base_excl]
        print(f"Shape: {df_race.shape} | Race: 0(Black)={(df_race['race_binary']==0).sum()}, 1(White)={(df_race['race_binary']==1).sum()}")
        return df_race, features, 'race_binary'
    elif mode == 'combined':
        features = [c for c in df_race.columns if c not in base_excl]
        print(f"Shape: {df_race.shape} | Combined:\n{df_race['combined'].value_counts().sort_index()}")
        print("0=Black-Female, 1=Black-Male, 2=White-Female, 3=White-Male")
        return df_race, features, 'combined'
    else:
        raise ValueError(f"mode must be 'gender'/'race'/'combined', got '{mode}'")


def load_dutch(mode='gender'):
    """
    D7: Dutch Census 2001
    mode: 'gender' only (no race attribute in this dataset)
    sex: 1=male, 2=female → sex_binary: 1=male, 0=female
    """
    print("\n" + "="*80)
    print(f"Loading Dutch Census Dataset (mode={mode})")
    print("="*80)
    if mode != 'gender':
        raise ValueError("Dutch Census only supports mode='gender' (no race attribute)")

    url = ('https://raw.githubusercontent.com/tailequy/fairness_dataset'
           '/main/Dutch_census/dutch_census_2001.arff')
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data, meta = arff.loadarff(io.StringIO(resp.text))
        df = pd.DataFrame(data)
        for col in df.select_dtypes(['object']).columns:
            df[col] = df[col].str.decode('utf-8')
        print(f"Loaded from URL. Shape: {df.shape}")
    except Exception as e:
        print(f"URL failed ({e}), trying local file ./data/dutch/dutch_census_2001.csv")
        df = pd.read_csv('./data/dutch/dutch_census_2001.csv')

    sex_col = [c for c in df.columns if c.lower() == 'sex'][0]
    unique_vals = set(df[sex_col].astype(str).unique())
    if unique_vals.issubset({'1', '2'}):
        df['sex_binary'] = (df[sex_col].astype(str) == '1').astype(int)
    else:
        df['sex_binary'] = (df[sex_col].str.lower().isin(['male', 'm', '1'])).astype(int)

    target_col = next((c for c in df.columns if c.lower() in ['occupation', 'label', 'class']), None)
    excl = [sex_col, 'sex_binary'] + ([target_col] if target_col else [])
    features = [c for c in df.columns if c not in excl]

    print(f"Gender: 0(F)={(df['sex_binary']==0).sum()}, 1(M)={(df['sex_binary']==1).sum()}")
    return df, features, 'sex_binary'


def load_meps(mode='gender'):
    print("\n" + "="*80)
    print(f"Loading MEPS Panel 20 (2016) Dataset (mode={mode})")
    print("="*80)

    import aif360
    meps_dir = os.path.join(os.path.dirname(aif360.__file__), 'data', 'raw', 'meps')
    df = pd.read_csv(meps_dir + r'/h192.csv', low_memory=False)
    print(f"Loaded h192.csv. Shape: {df.shape}")

    keep_cols = ['AGE31X','AGE42X','AGE53X','SEX','RACEV1X','MARRY31X','MARRY42X','MARRY53X',
                 'EDUCYR','HIDEG','FTSTU31X','FTSTU42X','FTSTU53X','ACTDTY31','ACTDTY42','ACTDTY53',
                 'HONRDC31','HONRDC42','HONRDC53','WRGFD31X','WRGFD42X','WRGFD53X',
                 'TTLP16X','POVCAT16','INSCOV16']
    available = [c for c in keep_cols if c in df.columns]
    print(f"Available key columns: {len(available)}/{len(keep_cols)}")
    df = df[available].dropna()
    print(f"Shape after dropna: {df.shape}")

    df['sex_binary']  = (df['SEX'] == 1).astype(int)
    df['race_binary'] = (df['RACEV1X'] == 1).astype(int)
    df['combined']    = df['race_binary'] * 2 + df['sex_binary']

    print(f"Sex distribution: {df['SEX'].value_counts().to_dict()}")
    print(f"Race distribution: {df['RACEV1X'].value_counts().to_dict()}")

    base_excl = ['SEX', 'RACEV1X', 'sex_binary', 'race_binary', 'combined']
    features = [c for c in df.columns if c not in base_excl]

    if mode == 'gender':
        print(f"Gender: 0(F)={(df['sex_binary']==0).sum()}, 1(M)={(df['sex_binary']==1).sum()}")
        return df, features, 'sex_binary'
    elif mode == 'race':
        print(f"Race: 0(NonWhite)={(df['race_binary']==0).sum()}, 1(White)={(df['race_binary']==1).sum()}")
        return df, features, 'race_binary'
    elif mode == 'combined':
        print(f"Combined:\n{df['combined'].value_counts().sort_index()}")
        print("0=NonWhite-Female, 1=NonWhite-Male, 2=White-Female, 3=White-Male")
        return df, features, 'combined'
    else:
        raise ValueError(f"mode must be gender/race/combined, got '{mode}'")

In [8]:
## Cell 4: Preprocessing

In [9]:
def preprocess_dataset(df, feature_cols, sensitive_col, max_unique_ratio=0.5,
                        numeric_convert_threshold=0.95, datetime_convert_threshold=0.95):
    import re
    X_df = df[feature_cols].copy()
    n = len(X_df)
    id_name_pattern = re.compile(r'(name|first|last|dob)', re.IGNORECASE)

    num_cols = []
    cat_cols = []
    excluded_cols = []

    for col in feature_cols:
        if X_df[col].dtype != object:
            num_cols.append(col)
            continue

        # recover object columns that are actually numeric, mistyped as string
        converted_num = pd.to_numeric(X_df[col], errors='coerce')
        if converted_num.notna().mean() >= numeric_convert_threshold:
            X_df[col] = converted_num
            num_cols.append(col)
            continue

        # exclude known identifier style fields, name, first, last, dob, by column name
        if id_name_pattern.search(str(col)):
            excluded_cols.append(col)
            continue

        # exclude columns that are almost entirely parseable as dates, these are
        # not nominal categories and should not be One Hot Encoded as such
        converted_dt = pd.to_datetime(X_df[col], errors='coerce')
        if converted_dt.notna().mean() >= datetime_convert_threshold:
            excluded_cols.append(col)
            continue

        # exclude remaining high cardinality object columns, id, case number,
        # free text, and similar fields
        unique_ratio = X_df[col].nunique() / n
        if unique_ratio > max_unique_ratio:
            excluded_cols.append(col)
            continue

        cat_cols.append(col)

    # excluded_cols is intentionally not printed, inspect it directly if needed

    num_part = X_df[num_cols].values.astype(float) if num_cols else np.zeros((len(X_df), 0))
    if cat_cols:
        ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        cat_part = ohe.fit_transform(X_df[cat_cols].astype(str))
    else:
        cat_part = np.zeros((len(X_df), 0))
    X = np.hstack([num_part, cat_part])
    if np.any(np.isnan(X)) or np.any(np.isinf(X)):
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    X = StandardScaler().fit_transform(X)
    sensitive_attr = df[sensitive_col].values.astype(int)
    print(f"  Final shape: {X.shape}")
    print(f"  Sensitive groups: {dict(zip(*np.unique(sensitive_attr, return_counts=True)))}")
    return X, sensitive_attr


In [10]:
## Cell 5: Load Datasets — Part B (8 configs: diabetes×3, dutch×1, meps×3)

In [11]:
# =============================================================================
# SECTION 4: LOAD DATASETS — Part B (8 configurations)
# diabetes×3, dutch×1, meps×3
# =============================================================================
print("\n" + "="*80)
print("LOADING DATASETS — PART B")
print("="*80)
datasets = {}

# --- Diabetes: gender / race / combined ---
for mode in ['gender', 'race', 'combined']:
    name = f'diabetes_{mode}'
    try:
        df, features, sensitive_col = load_diabetes(mode=mode)
        X, sensitive = preprocess_dataset(df, features, sensitive_col)
        datasets[name] = {'X': X, 'sensitive': sensitive,
                          'n_samples': X.shape[0], 'n_features': X.shape[1]}
        print(f"✓ {name}: {X.shape}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# --- Dutch Census: gender only ---
try:
    df, features, sensitive_col = load_dutch(mode='gender')
    X, sensitive = preprocess_dataset(df, features, sensitive_col)
    datasets['dutch_gender'] = {'X': X, 'sensitive': sensitive,
                                'n_samples': X.shape[0], 'n_features': X.shape[1]}
    print(f"✓ dutch_gender: {X.shape}")
except Exception as e:
    print(f"✗ dutch_gender: {e}")

# --- MEPS: gender / race / combined ---
for mode in ['gender', 'race', 'combined']:
    name = f'meps_{mode}'
    try:
        df, features, sensitive_col = load_meps(mode=mode)
        X, sensitive = preprocess_dataset(df, features, sensitive_col)
        datasets[name] = {'X': X, 'sensitive': sensitive,
                          'n_samples': X.shape[0], 'n_features': X.shape[1]}
        print(f"✓ {name}: {X.shape}")
    except Exception as e:
        print(f"✗ {name}: {e}")

print("\n" + "="*80)
print("DATASET SUMMARY")
print("="*80)
for name, data in datasets.items():
    print(f"  {name:30s}: {data['n_samples']:>7} samples, {data['n_features']:>4} features")

dataset_names = list(datasets.keys())
print(f"\nTotal configurations loaded: {len(dataset_names)}")
# Expected 8: diabetes×3, dutch×1, meps×3 (note: dutch has no race attribute)



LOADING DATASETS — PART B

Loading Diabetes 130-US Dataset (mode=gender)
Shape: (99492, 45) | Gender: 0(F)=53575, 1(M)=45917
  Final shape: (99492, 892)
  Sensitive groups: {0: 53575, 1: 45917}
✓ diabetes_gender: (99492, 892)

Loading Diabetes 130-US Dataset (mode=race)
Shape: (95309, 47) | Race: 0(Black)=19210, 1(White)=76099
  Final shape: (95309, 888)
  Sensitive groups: {0: 19210, 1: 76099}
✓ diabetes_race: (95309, 888)

Loading Diabetes 130-US Dataset (mode=combined)
Shape: (95309, 47) | Combined:
combined
0    11728
1     7482
2    39689
3    36410
Name: count, dtype: int64
0=Black-Female, 1=Black-Male, 2=White-Female, 3=White-Male
  Final shape: (95309, 888)
  Sensitive groups: {0: 11728, 1: 7482, 2: 39689, 3: 36410}
✓ diabetes_combined: (95309, 888)

Loading Dutch Census Dataset (mode=gender)
Loaded from URL. Shape: (60420, 12)
Gender: 0(F)=30273, 1(M)=30147
  Final shape: (60420, 10)
  Sensitive groups: {0: 30273, 1: 30147}
✓ dutch_gender: (60420, 10)

Loading MEPS Panel 20 (

In [12]:
## Cell 6: Clustering Algorithms

In [13]:
# =============================================================================
# SECTION 6: CLUSTERING ALGORITHMS (create_fairlets, bfkm_clustering, PostProcessingNFP, PostProcessingGini generalized to multi-group -- Reviewer 3, Comment 1)
# =============================================================================

def standard_kmeans(X, k, random_state=42):
    km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
    return km.fit_predict(X), km.cluster_centers_


# ORIGINAL (Reviewer 3, Comment 1 -- kept for reference, not executed):
# def create_fairlets(X, sensitive_attr, p, q):
#     n = len(X)
#     g0, g1 = np.where(sensitive_attr == 0)[0], np.where(sensitive_attr == 1)[0]
#     fairlets, fairlet_labels = [], np.full(n, -1)
#     used0, used1, fid = set(), set(), 0
#     while len(used0)+p <= len(g0) and len(used1)+q <= len(g1):
#         s0 = np.random.choice([i for i in g0 if i not in used0], p, replace=False)
#         s1 = np.random.choice([i for i in g1 if i not in used1], q, replace=False)
#         members = np.concatenate([s0, s1])
#         fairlets.append(members)
#         fairlet_labels[members] = fid
#         used0.update(s0); used1.update(s1); fid += 1
#     rem = np.where(fairlet_labels == -1)[0]
#     if len(rem):
#         fairlets.append(rem); fairlet_labels[rem] = fid
#     return fairlets, fairlet_labels

# Fairlet decomposition, generalised to multiple groups (our own extension).
# Chierichetti et al. (NeurIPS 2017) define fairlets for two groups only; no
# published multi-group fairlet algorithm exists. Here every fairlet contains
# points from every group, with each group's share set by rounding its count
# relative to the smallest group (in the spirit of Bera et al., NeurIPS 2019).
# The same rule is used for two or more groups.
# Limitation (disclosed in the manuscript): the original (b, r)-fairlet uses
# the exact group ratio in lowest terms; we use a rounded approximation to keep
# fairlets small (e.g. a 60:40 split becomes 2:1 rather than 3:2), because an
# exact reduction of real counts (e.g. gcd(21790, 10771) = 1) would require
# impractically large fairlets.
# `p`, `q` are unused and kept only for backward compatibility.


def create_fairlets(X, sensitive_attr, p=1, q=1):
    n = len(X)
    S = np.asarray(sensitive_attr)
    groups = np.unique(S)

    group_idxs = {g: np.where(S == g)[0] for g in groups}
    counts = {g: len(group_idxs[g]) for g in groups}
    min_count = max(1, min(counts.values()))
    unit_sizes = {g: max(1, round(counts[g] / min_count)) for g in groups}

    fairlets, fairlet_labels = [], np.full(n, -1)
    unused = {g: np.ones(counts[g], dtype=bool) for g in groups} 
    fid = 0
    while True:
        members, ok = [], True
        for g in groups:
            avail = group_idxs[g][unused[g]]   
            if len(avail) < unit_sizes[g]:
                ok = False
                break
            sel = np.random.choice(avail, unit_sizes[g], replace=False)
            members.append(sel)
            unused[g][np.searchsorted(group_idxs[g], sel)] = False
        if not ok:
            break
        members = np.concatenate(members)
        fairlets.append(members)
        fairlet_labels[members] = fid
        fid += 1
    rem = np.where(fairlet_labels == -1)[0]
    if len(rem):
        fairlets.append(rem)
        fairlet_labels[rem] = fid
    return fairlets, fairlet_labels


def fairlet_clustering(X, sensitive_attr, k, p=1, q=1):
    fairlets, _ = create_fairlets(X, sensitive_attr, p, q)
    centers_f = np.array([np.mean(X[f], axis=0) for f in fairlets])
    fc_labels = KMeans(n_clusters=min(k, len(centers_f)), random_state=42, n_init=10).fit_predict(centers_f) \
        if len(centers_f) >= k else np.arange(len(centers_f))
    labels = np.zeros(len(X), dtype=int)
    for i, cl in enumerate(fc_labels):
        labels[fairlets[i]] = cl
    centers = np.array([np.mean(X[labels == i], axis=0) if (labels == i).any()
                         else np.zeros(X.shape[1]) for i in range(k)])
    return labels, centers


# ORIGINAL (Reviewer 3, Comment 1 & 7 -- kept for reference, not executed):
# def bfkm_clustering(X, sensitive_attr, k, max_iter=100):
#     centers = X[np.random.choice(len(X), k, replace=False)].copy()
#     for _ in range(max_iter):
#         old = centers.copy()
#         dists = cdist(X, centers)
#         labels = np.argmin(dists, axis=1)
#         t0, t1 = (sensitive_attr == 0).sum(), (sensitive_attr == 1).sum()
#         for c in range(k):
#             mask = labels == c
#             s = sensitive_attr[mask]
#             n0, n1 = (s == 0).sum(), (s == 1).sum()
#             if t0 > 0 and t1 > 0:
#                 ratio_exp = t1 / t0
#                 ratio_act = n1 / max(n0, 1)
#                 if abs(ratio_act - ratio_exp) > 0.3:
#                     idxs = np.where(mask)[0]
#                     grp = 1 if ratio_act > ratio_exp else 0
#                     g_idxs = idxs[sensitive_attr[idxs] == grp]
#                     if len(g_idxs):
#                         far = g_idxs[np.argmax(dists[g_idxs, c])]
#                         sc = np.argsort(dists[far])
#                         labels[far] = sc[1] if sc[1] != c else sc[2]
#         for i in range(k):
#             pts = X[labels == i]
#             if len(pts): centers[i] = pts.mean(0)
#         if np.allclose(centers, old, rtol=1e-4): break
#     return labels, centers

# FIXED (Reviewer 3, Comment 1 & 7): generalized from binary (groups 0/1
# hardcoded) to any number of groups, matching the demographic-parity
# constraint of the original method (Pan, Zhong & Qian, 2023):
# P(cluster=j | group=l) = P(cluster=j) for every cluster j and group l --
# equivalently, each group's within-cluster proportion should match its
# global proportion. The solver itself remains the same greedy heuristic
# (find the cluster's most over-represented group relative to its global
# share, i.e. the largest POSITIVE deviation, and move its farthest point
# out; under-represented groups are never moved out), not the paper's exact
# Lagrangian-relaxed alternating optimization, and without its cluster-size
# balance term -- generalized in what it checks (all groups vs. a global
# reference), not in how it solves the optimization.

def bfkm_clustering(X, sensitive_attr, k, max_iter=100):
    centers = X[np.random.choice(len(X), k, replace=False)].copy()
    S = np.asarray(sensitive_attr)
    groups = np.unique(S)
    global_props = {g: (S == g).sum() / len(S) for g in groups}
    for _ in range(max_iter):
        old = centers.copy()
        dists = cdist(X, centers)
        labels = np.argmin(dists, axis=1)
        for c in range(k):
            mask = labels == c
            n_c = mask.sum()
            if n_c == 0:
                continue
            s = S[mask]
            worst_g, worst_dev = None, 0.0
            for g in groups:
                prop_c = (s == g).sum() / n_c
                dev = prop_c - global_props[g]
                if dev > worst_dev:
                    worst_g, worst_dev = g, dev
            if worst_g is not None and worst_dev > 0.3:
                idxs = np.where(mask)[0]
                g_idxs = idxs[S[idxs] == worst_g]
                if len(g_idxs):
                    far = g_idxs[np.argmax(dists[g_idxs, c])]
                    sc = np.argsort(dists[far])
                    labels[far] = sc[1] if sc[1] != c else sc[2]
        for i in range(k):
            pts = X[labels == i]
            if len(pts): centers[i] = pts.mean(0)
        if np.allclose(centers, old, rtol=1e-4): break
    return labels, centers


# ORIGINAL (runtime bottleneck -- kept for reference, not executed):
# class OptimizedFairCentroid:
#     def __init__(self, n_clusters=6, max_iter=50, min_iter=10,
#                  tol=0.005, random_state=0, candidate_fraction=0.05):
#         self.n_clusters = n_clusters
#         self.max_iter = max_iter; self.min_iter = min_iter
#         self.tol = tol; self.random_state = random_state
#         self.candidate_fraction = candidate_fraction
#         self.labels_ = self.cluster_centers_ = self.inertia_ = None; self.n_iter_ = 0
#
#     def _compute_cluster_fairness_fast(self, X_c, S_c, centroid):
#         if len(X_c) == 0: return 0.0
#         uq = np.unique(S_c)
#         if len(uq) < 2: return 0.0
#         d2 = np.sum((X_c - centroid)**2, axis=1)
#         gm = [d2[S_c == g].mean() for g in uq if (S_c == g).sum() > 0]
#         return max(gm) - min(gm) if len(gm) >= 2 else 0.0
#
#     def _precompute_cluster_stats(self, X, labels, S, centers):
#         stats = {}
#         for c in range(self.n_clusters):
#             mask = labels == c; n_c = mask.sum()
#             if n_c == 0:
#                 stats[c] = {'n': 0, 'centroid': centers[c], 'fairness': 0.0, 'X': np.array([]), 'S': np.array([])}
#             else:
#                 Xc, Sc = X[mask], S[mask]
#                 stats[c] = {'n': n_c, 'centroid': centers[c],
#                             'fairness': self._compute_cluster_fairness_fast(Xc, Sc, centers[c]),
#                             'X': Xc, 'S': Sc}
#         return stats
#
#     def _find_candidates(self, X, labels, centers, n_candidates):
#         n = len(X)
#         nc = max(1, int(n * self.candidate_fraction))
#         dists = np.array([np.linalg.norm(X[i] - centers[labels[i]]) for i in range(n)])
#         return np.argsort(-dists)[:nc]
#
#     def fit(self, X, sensitive_attr):
#         S = np.array(sensitive_attr if not hasattr(sensitive_attr, 'values') else sensitive_attr.values)
#         km = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
#         self.labels_ = km.fit_predict(X)
#         centers = km.cluster_centers_.copy()
#         for it in range(self.max_iter):
#             stats = self._precompute_cluster_stats(X, self.labels_, S, centers)
#             cands = self._find_candidates(X, self.labels_, centers, int(len(X)*self.candidate_fraction))
#             improved = False
#             for idx in cands:
#                 cur = self.labels_[idx]
#                 cur_fair = stats[cur]['fairness']
#                 best_c, best_delta = cur, 0
#                 for c2 in range(self.n_clusters):
#                     if c2 == cur: continue
#                     new_labels = self.labels_.copy()
#                     new_labels[idx] = c2
#                     new_centers = centers.copy()
#                     for ci in [cur, c2]:
#                         pts = X[new_labels == ci]
#                         if len(pts): new_centers[ci] = pts.mean(0)
#                     new_stats = self._precompute_cluster_stats(X, new_labels, S, new_centers)
#                     delta = (stats[cur]['fairness'] + stats[c2]['fairness']) - \
#                             (new_stats[cur]['fairness'] + new_stats[c2]['fairness'])
#                     if delta > best_delta:
#                         best_delta, best_c = delta, c2
#                 if best_c != cur:
#                     self.labels_[idx] = best_c
#                     for ci in [cur, best_c]:
#                         pts = X[self.labels_ == ci]
#                         if len(pts): centers[ci] = pts.mean(0)
#                     stats = self._precompute_cluster_stats(X, self.labels_, S, centers)
#                     improved = True
#             if it >= self.min_iter and not improved: break
#         self.cluster_centers_ = centers
#         self.n_iter_ = it + 1
#         return self

# CCF (Cluster-level Centroid Fairness), optimised implementation.
# Same algorithm as the original version (kept above, commented out): standard
# K-means (k-means++, n_init=10), then greedy local-search reassignment of the
# 5% of points farthest from their centroid, each moved to the cluster that most
# reduces the per-cluster disparity in mean squared distance-to-centroid between
# sensitive groups; stops when there is no gain after min_iter, or at max_iter.
# Only the evaluation of a candidate move is faster: per-(cluster, group) counts,
# coordinate sums and squared-norm sums give the post-move fairness in closed form,
#     mean ||x - mu||^2 = (sum ||x||^2 - 2 mu . sum x + n ||mu||^2) / n,
# instead of recomputing all clusters over the full data (O(groups*d) vs O(n*d*k)).
# Statistics are recomputed from the data at the start of every iteration.
# Verified to give labels identical to the original implementation.

class OptimizedFairCentroid:
    def __init__(self, n_clusters=6, max_iter=50, min_iter=10,
                 tol=0.005, random_state=0, candidate_fraction=0.05):
        self.n_clusters = n_clusters
        self.max_iter = max_iter; self.min_iter = min_iter
        self.tol = tol; self.random_state = random_state
        self.candidate_fraction = candidate_fraction
        self.labels_ = self.cluster_centers_ = self.inertia_ = None; self.n_iter_ = 0

    # unchanged from the original implementation
    def _compute_cluster_fairness_fast(self, X_c, S_c, centroid):
        if len(X_c) == 0: return 0.0
        uq = np.unique(S_c)
        if len(uq) < 2: return 0.0
        d2 = np.sum((X_c - centroid)**2, axis=1)
        gm = [d2[S_c == g].mean() for g in uq if (S_c == g).sum() > 0]
        return max(gm) - min(gm) if len(gm) >= 2 else 0.0

    # unchanged from the original implementation
    def _find_candidates(self, X, labels, centers, n_candidates):
        n = len(X)
        nc = max(1, int(n * self.candidate_fraction))
        dists = np.array([np.linalg.norm(X[i] - centers[labels[i]]) for i in range(n)])
        return np.argsort(-dists)[:nc]

    @staticmethod
    def _fair_from_stats(cnt_c, sx_c, sq_c, mu):
        # same quantity as _compute_cluster_fairness_fast, from sufficient statistics
        present = cnt_c > 0
        if present.sum() < 2:
            return 0.0
        n = cnt_c[present]
        gm = (sq_c[present] - 2.0 * (sx_c[present] @ mu) + n * (mu @ mu)) / n
        return float(gm.max() - gm.min())

    def fit(self, X, sensitive_attr):
        S = np.array(sensitive_attr if not hasattr(sensitive_attr, 'values') else sensitive_attr.values)
        km = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
        self.labels_ = km.fit_predict(X)
        centers = km.cluster_centers_.copy()
        labels = self.labels_
        k = self.n_clusters
        groups = np.unique(S)
        g_idx = np.searchsorted(groups, S)
        G = len(groups)
        sqn = np.einsum('ij,ij->i', X, X)
        touched = np.zeros(k, dtype=bool)

        for it in range(self.max_iter):
            # --- iteration start: recompute directly from the data (as original) ---
            for c in np.where(touched)[0]:
                pts = X[labels == c]
                if len(pts): centers[c] = pts.mean(0)
            touched[:] = False
            cnt = np.zeros((k, G)); sx = np.zeros((k, G, X.shape[1])); sq = np.zeros((k, G))
            fair = np.zeros(k)
            for c in range(k):
                mask = labels == c
                if mask.any():
                    fair[c] = self._compute_cluster_fairness_fast(X[mask], S[mask], centers[c])
                    for g in range(G):
                        m = mask & (g_idx == g)
                        cnt[c, g] = m.sum(); sx[c, g] = X[m].sum(0); sq[c, g] = sqn[m].sum()

            cands = self._find_candidates(X, labels, centers, int(len(X)*self.candidate_fraction))
            improved = False
            for idx in cands:
                cur, g, x, q = labels[idx], g_idx[idx], X[idx], sqn[idx]
                # cluster `cur` after removing the point
                cnt_a = cnt[cur].copy(); cnt_a[g] -= 1
                sx_a = sx[cur].copy();   sx_a[g] -= x
                sq_a = sq[cur].copy();   sq_a[g] -= q
                n_a = cnt_a.sum()
                mu_a = sx_a.sum(0) / n_a if n_a > 0 else centers[cur]
                f_a = self._fair_from_stats(cnt_a, sx_a, sq_a, mu_a) if n_a > 0 else 0.0
                best_c, best_delta = cur, 0
                for c2 in range(k):
                    if c2 == cur: continue
                    # cluster `c2` after adding the point
                    cnt_b = cnt[c2].copy(); cnt_b[g] += 1
                    sx_b = sx[c2].copy();   sx_b[g] += x
                    sq_b = sq[c2].copy();   sq_b[g] += q
                    mu_b = sx_b.sum(0) / cnt_b.sum()
                    delta = (fair[cur] + fair[c2]) - \
                            (f_a + self._fair_from_stats(cnt_b, sx_b, sq_b, mu_b))
                    if delta > best_delta:
                        best_delta, best_c = delta, c2
                if best_c != cur:
                    labels[idx] = best_c
                    cnt[cur, g] -= 1; sx[cur, g] -= x; sq[cur, g] -= q
                    cnt[best_c, g] += 1; sx[best_c, g] += x; sq[best_c, g] += q
                    for ci in (cur, best_c):
                        n_ci = cnt[ci].sum()
                        if n_ci > 0:
                            centers[ci] = sx[ci].sum(0) / n_ci
                            touched[ci] = True
                        fair[ci] = self._fair_from_stats(cnt[ci], sx[ci], sq[ci], centers[ci]) if n_ci > 0 else 0.0
                    improved = True
            if it >= self.min_iter and not improved: break

        # final centroids recomputed directly from the data, as in the original
        for c in np.where(touched)[0]:
            pts = X[labels == c]
            if len(pts): centers[c] = pts.mean(0)
        self.cluster_centers_ = centers
        self.n_iter_ = it + 1
        return self


def fair_centroid_clustering(X, sensitive_attr, k, max_iter=20, min_iter=5, random_state=0):
    model = OptimizedFairCentroid(n_clusters=k, max_iter=max_iter,
                                   min_iter=min_iter, random_state=random_state)
    model.fit(X, sensitive_attr)
    return model.labels_, model.cluster_centers_

# ORIGINAL (Reviewer 3, Comment 1 -- kept for reference, not executed):
# class PostProcessingNFP:
#     def __init__(self, n_clusters=6, balance_tolerance=0.05, max_iter=30, random_state=0):
#         self.n_clusters = n_clusters; self.balance_tolerance = balance_tolerance
#         self.max_iter = max_iter; self.random_state = random_state
#         self.labels_ = self.cluster_centers_ = None
#
#     def fit(self, X, sensitive_attr):
#         S = np.array(sensitive_attr)
#         km = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
#         self.labels_ = km.fit_predict(X)
#         centers = km.cluster_centers_.copy()
#         t0, t1 = (S == 0).sum(), (S == 1).sum()
#         global_ratio = t1 / max(t0, 1)
#         for _ in range(self.max_iter):
#             changed = False
#             ratios = []
#             for c in range(self.n_clusters):
#                 mask = self.labels_ == c
#                 s = S[mask]
#                 ratios.append((s == 1).sum() / max((s == 0).sum(), 1))
#             max_c, min_c = np.argmax(ratios), np.argmin(ratios)
#             if abs(ratios[max_c] - global_ratio) <= self.balance_tolerance: break
#             mask_max = np.where((self.labels_ == max_c) & (S == 1))[0]
#             if len(mask_max) == 0: break
#             dists_to_min = np.linalg.norm(X[mask_max] - centers[min_c], axis=1)
#             nearest = mask_max[np.argmin(dists_to_min)]
#             self.labels_[nearest] = min_c
#             for ci in [max_c, min_c]:
#                 pts = X[self.labels_ == ci]
#                 if len(pts): centers[ci] = pts.mean(0)
#             changed = True
#             if not changed: break
#         self.cluster_centers_ = centers
#         return self

# FIXED (Reviewer 3, Comment 1): generalized from binary (groups 0/1
# hardcoded) to any number of groups. Previously this class only ever
# compared group-1-vs-group-0 ratios, so for _combined (4-group)
# configurations groups 2 and 3 were never checked during reassignment.
# Now finds the (cluster, group) pair with the worst positive deviation of
# within-cluster proportion from global proportion (the same
# demographic-parity criterion used for the generalized BFKM above), and
# reassigns that group's nearest-to-target-centroid point to whichever
# other cluster most under-represents that same group -- preserving the
# original "nearest foreign point" mechanism, generalized to multiple
# groups instead of a single binary comparison.

class PostProcessingNFP:
    def __init__(self, n_clusters=6, balance_tolerance=0.05, max_iter=30, random_state=0):
        self.n_clusters = n_clusters; self.balance_tolerance = balance_tolerance
        self.max_iter = max_iter; self.random_state = random_state
        self.labels_ = self.cluster_centers_ = None

    def fit(self, X, sensitive_attr):
        S = np.array(sensitive_attr)
        groups = np.unique(S)
        global_props = {g: (S == g).sum() / len(S) for g in groups}
        km = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
        self.labels_ = km.fit_predict(X)
        centers = km.cluster_centers_.copy()
        for _ in range(self.max_iter):
            worst_c, worst_g, worst_dev = None, None, 0.0
            for c in range(self.n_clusters):
                mask = self.labels_ == c
                n_c = mask.sum()
                if n_c == 0:
                    continue
                s = S[mask]
                for g in groups:
                    dev = (s == g).sum() / n_c - global_props[g]
                    if dev > worst_dev:
                        worst_c, worst_g, worst_dev = c, g, dev
            if worst_c is None or worst_dev <= self.balance_tolerance:
                break
            under_c, under_dev = None, 0.0
            for c in range(self.n_clusters):
                if c == worst_c:
                    continue
                mask = self.labels_ == c
                n_c = mask.sum()
                if n_c == 0:
                    continue
                dev = global_props[worst_g] - (S[mask] == worst_g).sum() / n_c
                if dev > under_dev:
                    under_c, under_dev = c, dev
            if under_c is None:
                break
            cand = np.where((self.labels_ == worst_c) & (S == worst_g))[0]
            if len(cand) == 0:
                break
            dists_to_target = np.linalg.norm(X[cand] - centers[under_c], axis=1)
            nearest = cand[np.argmin(dists_to_target)]
            self.labels_[nearest] = under_c
            for ci in [worst_c, under_c]:
                pts = X[self.labels_ == ci]
                if len(pts): centers[ci] = pts.mean(0)
        self.cluster_centers_ = centers
        return self


# ORIGINAL (Reviewer 3, Comment 1 -- kept for reference, not executed):
# class PostProcessingGini:
#     def __init__(self, n_clusters=6, k_neighbors=10, balance_tolerance=0.05, max_iter=30, random_state=0):
#         self.n_clusters = n_clusters; self.k_neighbors = k_neighbors
#         self.balance_tolerance = balance_tolerance; self.max_iter = max_iter
#         self.random_state = random_state; self.labels_ = self.cluster_centers_ = None
#
#     def _compute_gini_coefficient(self, group_counts):
#         if sum(group_counts.values()) == 0: return 0
#         total = sum(group_counts.values())
#         probs = [v/total for v in group_counts.values()]
#         return 1 - sum(p**2 for p in probs)
#
#     def _compute_overall_gini(self, labels, S, k):
#         ginis = []
#         for c in range(k):
#             mask = labels == c
#             if mask.sum() == 0: continue
#             s = S[mask]
#             counts = {g: (s == g).sum() for g in np.unique(S)}
#             ginis.append(self._compute_gini_coefficient(counts))
#         return np.mean(ginis) if ginis else 0
#
#     def fit(self, X, sensitive_attr):
#         S = np.array(sensitive_attr)
#         km = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
#         self.labels_ = km.fit_predict(X)
#         centers = km.cluster_centers_.copy()
#         t0, t1 = (S == 0).sum(), (S == 1).sum()
#         global_ratio = t1 / max(t0, 1)
#         for _ in range(self.max_iter):
#             ratios = [(S[(self.labels_ == c) & True] == 1).sum() /
#                       max((S[self.labels_ == c] == 0).sum(), 1) for c in range(self.n_clusters)]
#             max_c, min_c = np.argmax(ratios), np.argmin(ratios)
#             if abs(ratios[max_c] - global_ratio) <= self.balance_tolerance: break
#             # Gini-guided selection
#             cands = np.where((self.labels_ == max_c) & (S == 1))[0]
#             if len(cands) == 0: break
#             ginis = []
#             for idx in cands:
#                 tmp = self.labels_.copy(); tmp[idx] = min_c
#                 ginis.append(self._compute_overall_gini(tmp, S, self.n_clusters))
#             best = cands[np.argmax(ginis)]
#             self.labels_[best] = min_c
#             for ci in [max_c, min_c]:
#                 pts = X[self.labels_ == ci]
#                 if len(pts): centers[ci] = pts.mean(0)
#         self.cluster_centers_ = centers
#         return self

# FIXED (Reviewer 3, Comment 1): Gini now chooses the move (source cluster,
# group, target cluster) from the cluster x group counts; distance chooses
# the point (nearest to the target centroid, same rule as PostProcessingNFP).
# Unlike PostProcessingNFP, which always moves the worst over-represented
# group to the cluster that most under-represents it, the move here is
# whichever one most increases the mean Gini impurity.
# Generalized from binary to any number of groups via np.unique(S): the
# original loop compared only group 1 vs group 0, so for _combined (4-group)
# configurations groups 2 and 3 were never considered.
# Stops at max deviation <= 0.05, no Gini gain, or max_iter.

class PostProcessingGini:
    def __init__(self, n_clusters=6, k_neighbors=10, balance_tolerance=0.05, max_iter=30, random_state=0):
        self.n_clusters = n_clusters; self.k_neighbors = k_neighbors
        self.balance_tolerance = balance_tolerance; self.max_iter = max_iter
        self.random_state = random_state; self.labels_ = self.cluster_centers_ = None
 
    @staticmethod
    def _mean_gini(counts):
        # counts: (n_clusters x n_groups); mean Gini impurity over non-empty clusters
        sizes = counts.sum(1)
        nonempty = sizes > 0
        if not nonempty.any(): return 0.0
        p = counts[nonempty] / sizes[nonempty, None]
        return float(np.mean(1 - (p ** 2).sum(1)))
 
    def fit(self, X, sensitive_attr):
        S = np.array(sensitive_attr)
        groups = np.unique(S)
        g_idx = np.searchsorted(groups, S)
        global_props = np.bincount(g_idx, minlength=len(groups)) / len(S)
        km = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
        self.labels_ = km.fit_predict(X)
        centers = km.cluster_centers_.copy()
        for _ in range(self.max_iter):
            counts = np.zeros((self.n_clusters, len(groups)))
            np.add.at(counts, (self.labels_, g_idx), 1)
            sizes = counts.sum(1, keepdims=True)
            dev = np.where(sizes > 0, counts / np.maximum(sizes, 1) - global_props, 0.0)
            if dev.max() <= self.balance_tolerance:
                break
            # Gini-guided choice of the move
            base = self._mean_gini(counts)
            best_gain, best_move = 1e-12, None
            for c in range(self.n_clusters):
                if sizes[c, 0] <= 1:
                    continue                      # never empty a cluster
                for g in range(len(groups)):
                    if counts[c, g] == 0:
                        continue
                    for c2 in range(self.n_clusters):
                        if c2 == c:
                            continue
                        counts[c, g] -= 1; counts[c2, g] += 1
                        gain = self._mean_gini(counts) - base
                        counts[c, g] += 1; counts[c2, g] -= 1
                        if gain > best_gain:
                            best_gain, best_move = gain, (c, g, c2)
            if best_move is None:
                break
            c, g, c2 = best_move
            # distance-guided choice of the point
            cand = np.where((self.labels_ == c) & (g_idx == g))[0]
            dists_to_target = np.linalg.norm(X[cand] - centers[c2], axis=1)
            nearest = cand[np.argmin(dists_to_target)]
            self.labels_[nearest] = c2
            for ci in [c, c2]:
                pts = X[self.labels_ == ci]
                if len(pts): centers[ci] = pts.mean(0)
        self.cluster_centers_ = centers
        return self

def postprocessing_clustering_nfp(X, sensitive_attr, k, max_iter=30, random_state=0):
    m = PostProcessingNFP(n_clusters=k, max_iter=max_iter, random_state=random_state)
    m.fit(X, sensitive_attr)
    return m.labels_, m.cluster_centers_


def postprocessing_clustering_gini(X, sensitive_attr, k, max_iter=30, random_state=0):
    m = PostProcessingGini(n_clusters=k, max_iter=max_iter, random_state=random_state)
    m.fit(X, sensitive_attr)
    return m.labels_, m.cluster_centers_
    

def rawlsian_kmeans(X, sensitive_attr, k, n_runs=30, delta=None, random_state=0):
    Xd = X.toarray() if hasattr(X, 'toarray') else X
    S = sensitive_attr.values if hasattr(sensitive_attr, 'values') else np.array(sensitive_attr)
    if delta is None: delta = np.sqrt(Xd.shape[1])
    best_score, best_labels, best_centers = -np.inf, None, None
    for seed in range(n_runs):
        km = KMeans(n_clusters=k, random_state=random_state+seed, n_init=1)
        labels = km.fit_predict(Xd); centers = km.cluster_centers_
        utils = np.maximum(0, 1 - np.linalg.norm(Xd - centers[labels], axis=1) / delta)
        gu = [utils[S == g].mean() for g in np.unique(S) if (S == g).sum() > 0]
        score = min(gu) if gu else 0
        if score > best_score:
            best_score, best_labels, best_centers = score, labels.copy(), centers.copy()
    return best_labels, best_centers

In [14]:
## Cell 7: Quality & Fairness Metrics

In [15]:
# =============================================================================
# SECTION 7: METRICS (Balance, SPD, Disparate_Impact, Entropy fixed to match Eq. 5-8 and generalized to multi-group -- Reviewer 3, Comment 1 & 2)
# =============================================================================

def calculate_quality_metrics(X, labels):
    k = len(np.unique(labels))
    centers = np.array([X[labels==i].mean(0) if (labels==i).any() else np.zeros(X.shape[1])
                         for i in range(k)])
    inertia = sum(np.sum((X[labels==i]-centers[i])**2) for i in range(k))
    sil = silhouette_score(X, labels) if 1 < k < len(X) else 0
    ch  = calinski_harabasz_score(X, labels) if 1 < k < len(X) else 0
    db  = davies_bouldin_score(X, labels) if 1 < k < len(X) else 0
    overall_mean = X.mean(0)
    wcss = sum(np.sum((X[labels==i]-centers[i])**2) for i in range(k))
    bcss = sum(len(X[labels==i])*np.sum((centers[i]-overall_mean)**2) for i in range(k))
    tss = np.sum((X-overall_mean)**2)
    vr = bcss/wcss if wcss > 0 else 0
    try:
        icd = np.mean(pdist(centers)) if k > 1 else 0
    except: icd = 0
    intra_d = sep = dunn = np.nan
    return {'Inertia': inertia, 'Silhouette': sil, 'Calinski_Harabasz': ch,
            'Davies_Bouldin': db, 'Dunn_Index': dunn, 'WCSS': wcss, 'BCSS': bcss,
            'TSS': tss, 'Variance_Ratio': vr, 'Inter_Cluster_Distance': icd,
            'Intra_Cluster_Distance': intra_d, 'Separation_Index': sep}


# ORIGINAL (Reviewer 3, Comment 1 & 2 -- kept for reference, not executed):
# def calculate_balance(labels, S):
#     balances = []
#     for c in np.unique(labels):
#         sc = S[labels==c]
#         n0, n1 = (sc==0).sum(), (sc==1).sum()
#         if n0 > 0 and n1 > 0:
#             balances.append(min(n0,n1)/max(n0,n1))
#     return np.mean(balances) if balances else 0.0
#
#
# def calculate_spd(labels, S):
#     t0, t1 = (S==0).sum(), (S==1).sum()
#     vals = []
#     for c in np.unique(labels):
#         mask = labels==c
#         p0 = (S[mask]==0).sum()/t0 if t0 > 0 else 0
#         p1 = (S[mask]==1).sum()/t1 if t1 > 0 else 0
#         vals.append(abs(p0-p1))
#     return np.mean(vals)
#
#
# def calculate_disparate_impact(labels, S):
#     t0, t1 = (S==0).sum(), (S==1).sum()
#     vals = []
#     for c in np.unique(labels):
#         mask = labels==c
#         p0 = (S[mask]==0).sum()/t0 if t0 > 0 else 0
#         p1 = (S[mask]==1).sum()/t1 if t1 > 0 else 0
#         if p0 > 0 and p1 > 0:
#             vals.append(min(p0,p1)/max(p0,p1))
#     return np.mean(vals) if vals else 0.0
#
#
# def calculate_entropy_metric(labels, S):
#     entropies = []
#     for c in np.unique(labels):
#         sc = S[labels==c]
#         n0, n1 = (sc==0).sum(), (sc==1).sum()
#         if n0 > 0 and n1 > 0:
#             p = np.array([n0, n1]) / len(sc)
#             entropies.append(entropy(p, base=2))
#     return np.mean(entropies) if entropies else 0.0

# FIXED (Reviewer 3, Comment 1): generalized from binary (groups 0/1
# hardcoded) to any number of groups via np.unique(S), so _combined
# (4-group) configurations are no longer silently reduced to a 2-group
# comparison.
# FIXED (Reviewer 3, Comment 2): Balance changed from a mean across
# clusters to the worst-cluster minimum, matching Eq. (7).

def calculate_balance(labels, S):
    groups = np.unique(S)
    balances = []
    for c in np.unique(labels):
        sc = S[labels == c]
        props = np.array([(sc == g).sum() for g in groups]) / max(len(sc), 1)
        balances.append(props.min() / props.max() if props.max() > 0 else 0.0)
    return min(balances) if balances else 0.0


# FIXED (Reviewer 3, Comment 1 & 2): generalized to any number of groups;
# changed from a mean of |p0-p1| between two hardcoded groups to the
# maximum absolute deviation of within-cluster group proportion from the
# group's global proportion, matching Eq. (5): SPD = max_{k,a} |P(a|Ck) - P(a)|
def calculate_spd(labels, S):
    groups = np.unique(S)
    global_props = {g: (S == g).sum() / len(S) for g in groups}
    vals = []
    for c in np.unique(labels):
        mask = labels == c
        n_c = mask.sum()
        if n_c == 0:
            continue
        for g in groups:
            p_a_given_ck = (S[mask] == g).sum() / n_c
            vals.append(abs(p_a_given_ck - global_props[g]))
    return max(vals) if vals else 0.0


# FIXED (Reviewer 3, Comment 1 & 2): generalized to any number of groups;
# now returns the RAW Disparate Impact value (ideal = 1), matching Eq. (6):
# DI = min_{k,a} P(a|Ck) / P(a). Previously this function silently mixed a
# binary min/max ratio with an aggregation elsewhere that treated smaller
# raw values as "better" -- direction is now handled explicitly via
# DI_dist = |DI - 1| wherever this value is used for ranking (see run_single
# below), rather than baked into this function.
def calculate_disparate_impact(labels, S):
    groups = np.unique(S)
    global_props = {g: (S == g).sum() / len(S) for g in groups}
    vals = []
    for c in np.unique(labels):
        mask = labels == c
        n_c = mask.sum()
        if n_c == 0:
            continue
        for g in groups:
            p_a = global_props[g]
            if p_a > 0:
                p_a_given_ck = (S[mask] == g).sum() / n_c
                vals.append(p_a_given_ck / p_a)
    return min(vals) if vals else 0.0


# FIXED (Reviewer 3, Comment 2): generalized to any number of groups (was
# hardcoded to n0, n1); pure clusters are now INCLUDED with entropy = 0
# rather than excluded, since a pure cluster is a real (maximally unequal)
# outcome that should count toward the average, not be skipped.
def calculate_entropy_metric(labels, S):
    groups = np.unique(S)
    entropies = []
    for c in np.unique(labels):
        sc = S[labels == c]
        if len(sc) == 0:
            continue
        counts = np.array([(sc == g).sum() for g in groups])
        p = counts / counts.sum()
        p = p[p > 0]  # drop zero-probability terms so entropy() doesn't warn; result is unaffected (0*log(0) := 0)
        entropies.append(entropy(p, base=2))  # = 0 for a pure cluster, now included rather than excluded
    return np.mean(entropies) if entropies else 0.0

## Checkpoint / Resume
Running the next cell loads the checkpoint; the k-selection and 30-run cells below then skip everything already finished.

In [17]:
# =============================================================================
# CHECKPOINT / RESUME — NEW
# -----------------------------------------------------------------------------
# k-selection: saved after every {dataset x method} (every dataset for Strategy 1).
# 30-run loop: saved after every single seed.
# If the run is interrupted (crash / power loss / kernel died), just re-run all
# cells from the top: finished parts are loaded from the checkpoint and skipped,
# and the loop continues from the seed where it stopped. Every seed calls
# np.random.seed(seed), so resumed results are identical to an uninterrupted run.
# To start over completely: set FRESH_START = True, run once (deletes the old
# checkpoint), then set it back to False.
# =============================================================================
import os, pickle, time

CKPT_PATH   = 'ckpt_S2B_kmf.pkl'   # checkpoint file for this notebook (same folder as the notebook)
FRESH_START = False

def _empty_ckpt():
    return {'ksel': {}, 'done': {}, 'partial': {}}

def save_ckpt(ck):
    # write to a temp file, then replace: a crash during saving cannot corrupt the old checkpoint
    tmp = CKPT_PATH + '.tmp'
    with open(tmp, 'wb') as f:
        pickle.dump(ck, f)
    os.replace(tmp, CKPT_PATH)

def load_ckpt():
    if FRESH_START and os.path.exists(CKPT_PATH):
        os.remove(CKPT_PATH)
        print(f"FRESH_START=True -> deleted old checkpoint {CKPT_PATH}")
    if os.path.exists(CKPT_PATH):
        with open(CKPT_PATH, 'rb') as f:
            ck = pickle.load(f)
        n_part = sum(len(p['seeds_done']) for p in ck['partial'].values())
        print(f"✓ Loaded checkpoint {CKPT_PATH}: k-selection done for {len(ck['ksel'])} item(s), "
              f"30-run done for {len(ck['done'])} dataset x method, "
              f"{len(ck['partial'])} partially done ({n_part} seed(s) in total)")
        return ck
    print(f"No checkpoint found ({CKPT_PATH}), starting from scratch")
    return _empty_ckpt()

ckpt = load_ckpt()


✓ Loaded checkpoint ckpt_S2B_kmf.pkl: k-selection done for 21 item(s), 30-run done for 1 dataset x method, 1 partially done (4 seed(s) in total)


In [18]:
## Cell 8: Majority Vote K-Selection

In [19]:
from collections import Counter
from sklearn.metrics import adjusted_rand_score
import time

# =============================================================================
# MAJORITY VOTE K SELECTION — PER ALGORITHM
# Fast algorithms (kmeans, fairlet, bfkm): 6 indices using the algorithm itself
# Note: elbow/gap/stability use KMeans for speed; silhouette/calinski/davies
#       use the actual algorithm to reflect its own cluster quality
# =============================================================================

def run_algo_at_k(X, sensitive, method, k):
    """Run one algorithm at one k, return labels."""
    np.random.seed(0)
    if method == 'kmeans':
        labels, _ = standard_kmeans(X, k, random_state=42)
    elif method == 'fairlet':
        labels, _ = fairlet_clustering(X, sensitive, k)
    elif method == 'bfkm':
        labels, _ = bfkm_clustering(X, sensitive, k)
    return labels

def silhouette_for_algo(X, sensitive, method, k_range=(2,11)):
    scores = []
    for k in range(*k_range):
        try:
            labels = run_algo_at_k(X, sensitive, method, k)
            scores.append(silhouette_score(X, labels) if len(np.unique(labels))>1 else -1)
        except: scores.append(-1)
    return list(range(*k_range))[np.argmax(scores)]

def calinski_for_algo(X, sensitive, method, k_range=(2,11)):
    scores = []
    for k in range(*k_range):
        try:
            labels = run_algo_at_k(X, sensitive, method, k)
            scores.append(calinski_harabasz_score(X, labels) if len(np.unique(labels))>1 else 0)
        except: scores.append(0)
    return list(range(*k_range))[np.argmax(scores)]

def davies_for_algo(X, sensitive, method, k_range=(2,11)):
    scores = []
    for k in range(*k_range):
        try:
            labels = run_algo_at_k(X, sensitive, method, k)
            scores.append(davies_bouldin_score(X, labels) if len(np.unique(labels))>1 else 999)
        except: scores.append(999)
    return list(range(*k_range))[np.argmin(scores)]

# Elbow, Gap, Stability still use KMeans (structural, not algorithm-specific)
def elbow_method(X, k_range=(2,11)):
    wcss = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(X).inertia_
            for k in range(*k_range)]
    d2 = np.diff(np.diff(wcss))
    return list(range(*k_range))[min(np.argmax(d2)+2, len(range(*k_range))-1)]

def gap_statistic(X, k_range=(2,11), n_refs=10):
    gaps = []
    for k in range(*k_range):
        wk   = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X).inertia_
        refs = [KMeans(n_clusters=k, random_state=None, n_init=10).fit(
                    np.random.uniform(X.min(0), X.max(0), X.shape)).inertia_
                for _ in range(n_refs)]
        gaps.append(np.log(np.mean(refs)) - np.log(wk))
    return list(range(*k_range))[np.argmax(gaps)]

def stability_method(X, k_range=(2,11), n_iter=10, sample_ratio=0.8):
    n = int(len(X)*sample_ratio); stab = []
    for k in range(*k_range):
        aris = []
        for _ in range(n_iter):
            i1 = np.random.choice(len(X), n, replace=False)
            i2 = np.random.choice(len(X), n, replace=False)
            common = np.intersect1d(i1, i2)
            if len(common) > k:
                l1 = KMeans(n_clusters=k, random_state=None, n_init=10).fit_predict(X[i1])
                l2 = KMeans(n_clusters=k, random_state=None, n_init=10).fit_predict(X[i2])
                # FIXED (speed only, identical result): position lookup instead of
                # np.where(i1 == x) for every common point (was O(n^2), ~5 h on Diabetes)
                pos1 = np.empty(len(X), dtype=np.int64); pos1[i1] = np.arange(n)
                pos2 = np.empty(len(X), dtype=np.int64); pos2[i2] = np.arange(n)
                c1 = pos1[common]
                c2 = pos2[common]
                aris.append(adjusted_rand_score(l1[c1], l2[c2]))
        stab.append(np.mean(aris) if aris else 0)
    return list(range(*k_range))[np.argmax(stab)]

# ORIGINAL (kept for reference, not executed):
# def majority_vote_k_for_method(X, sensitive, method, k_range=(2,11)):
#     """Majority vote k using the algorithm's own cluster quality where possible."""
#     k1 = elbow_method(X, k_range)
#     k2 = silhouette_for_algo(X, sensitive, method, k_range)
#     k3 = calinski_for_algo(X, sensitive, method, k_range)
#     k4 = davies_for_algo(X, sensitive, method, k_range)
#     k5 = gap_statistic(X, k_range)
#     k6 = stability_method(X, k_range)
#     votes  = [k1, k2, k3, k4, k5, k6]
#     cnt    = Counter(votes)
#     winner = cnt.most_common(1)[0][0]
#     return winner, {'elbow':k1,'silhouette':k2,'calinski':k3,
#                     'davies':k4,'gap':k5,'stability':k6,'majority':winner}

# =============================================================================
# FIXED (Section 3.4): elbow, Gap and stability are computed ONCE per dataset on
# standard K-means and shared by every method, in both the kmf and the ccf
# notebook (previously they were recomputed for every method, and because Gap
# and stability use random sampling, the same dataset could receive different
# structural votes for different methods, e.g. COMPAS stability = 2 for K-Means
# but 4 for Fairlet). The shared votes are stored in S2_structural_votes.pkl.
# If a kmf checkpoint already holds K-Means votes for a dataset, those are
# reused, so already-finished K-Means results keep exactly the same k.
# Silhouette, Calinski-Harabasz and Davies-Bouldin are still computed on each
# target algorithm's own clustering output.
# =============================================================================
import os
STRUCT_PATH = 'S2_structural_votes.pkl'      # shared by all four Strategy 2 notebooks
STRUCT_KEYS = ('elbow', 'gap', 'stability')
VOTE_ORDER  = ['elbow', 'silhouette', 'calinski', 'davies', 'gap', 'stability']

STRUCT = {}
if os.path.exists(STRUCT_PATH):
    with open(STRUCT_PATH, 'rb') as f:
        STRUCT = pickle.load(f)
# seed from K-Means votes already selected in the kmf notebooks (keeps finished K-Means k unchanged)
for _ck in ('ckpt_S2A_kmf.pkl', 'ckpt_S2B_kmf.pkl'):
    if os.path.exists(_ck):
        with open(_ck, 'rb') as f:
            _kmf = pickle.load(f)
        for (_ds, _m), _e in _kmf.get('ksel', {}).items():
            if _m == 'kmeans' and _ds not in STRUCT:
                STRUCT[_ds] = {c: _e['votes'][c] for c in STRUCT_KEYS}
# FIXED: elbow / Gap / stability are taken from Strategy 1,
# so the shared structural votes are identical in both strategies and K-Means
# receives the same k under Strategy 1 and Strategy 2
_n_s1 = 0
for _csv in ('optimal_k_summary_PartA.csv', 'optimal_k_summary_PartB.csv'):
    if os.path.exists(_csv):
        for _, _r in pd.read_csv(_csv).iterrows():
            STRUCT[_r['Dataset']] = {'elbow': int(_r['Elbow']), 'gap': int(_r['Gap']),
                                     'stability': int(_r['Stability'])}
            _n_s1 += 1
print(f"Structural votes taken from Strategy 1 for {_n_s1} dataset(s)")
_missing = [ds for ds in dataset_names if ds not in STRUCT]
assert not _missing, f"No Strategy 1 votes for {_missing}: run Strategy 1 first"
with open(STRUCT_PATH, 'wb') as f:
    pickle.dump(STRUCT, f)

def structural_votes(dataset_name, X, k_range=(2, 11)):
    """Elbow / Gap / stability on standard K-means, computed once per dataset."""
    if dataset_name not in STRUCT:
        STRUCT[dataset_name] = {'elbow':     elbow_method(X, k_range),
                                'gap':       gap_statistic(X, k_range),
                                'stability': stability_method(X, k_range)}
        with open(STRUCT_PATH, 'wb') as f:
            pickle.dump(STRUCT, f)
    return STRUCT[dataset_name]

def algo_votes(X, sensitive, method, k_range=(2, 11)):
    """Silhouette / CHI / DBI on the target algorithm's own labels.
    Same result as silhouette_for_algo + calinski_for_algo + davies_for_algo
    (run_algo_at_k is deterministic, np.random.seed(0)), but each k is
    clustered once instead of three times."""
    ks = list(range(*k_range))
    sil, chi, dbi = [], [], []
    for k in ks:
        try:
            labels = run_algo_at_k(X, sensitive, method, k)
        except Exception:
            sil.append(-1); chi.append(0); dbi.append(999); continue
        ok = len(np.unique(labels)) > 1
        try: sil.append(silhouette_score(X, labels) if ok else -1)
        except Exception: sil.append(-1)
        try: chi.append(calinski_harabasz_score(X, labels) if ok else 0)
        except Exception: chi.append(0)
        try: dbi.append(davies_bouldin_score(X, labels) if ok else 999)
        except Exception: dbi.append(999)
    return {'silhouette': ks[int(np.argmax(sil))],
            'calinski':   ks[int(np.argmax(chi))],
            'davies':     ks[int(np.argmin(dbi))]}

def majority_from_votes(votes):
    winner = Counter([votes[c] for c in VOTE_ORDER]).most_common(1)[0][0]
    return winner, {**{c: votes[c] for c in VOTE_ORDER}, 'majority': winner}

def majority_vote_k_for_method(X, sensitive, method, dataset_name, k_range=(2, 11)):
    votes = {**structural_votes(dataset_name, X, k_range),
             **algo_votes(X, sensitive, method, k_range)}
    return majority_from_votes(votes)


# Run per-algorithm k-selection
print("\n" + "="*80)
print("STRATEGY 2 — PER-ALGORITHM MAJORITY VOTE K SELECTION")
print("(silhouette/calinski/davies computed using each algorithm's own labels)")
print("="*80)

methods = ['kmeans', 'fairlet', 'bfkm']
majority_k_results = {}   # {dataset: {method: {'majority_k': k, 'votes': {...}}}}

for dataset_name in dataset_names:
    X   = datasets[dataset_name]['X']
    sen = datasets[dataset_name]['sensitive']
    majority_k_results[dataset_name] = {}
    print(f"\n{dataset_name.upper()}:")
    for method in methods:
        # NEW: checkpoint / resume -- k already selected for this dataset x method, load it
        if (dataset_name, method) in ckpt['ksel']:
            # FIXED (Section 3.4): re-derive k with the SHARED elbow/Gap/stability votes;
            # the stored Silhouette/CHI/DBI votes of this method are kept unchanged.
            old = ckpt['ksel'][(dataset_name, method)]
            k, votes = majority_from_votes({**old['votes'], **structural_votes(dataset_name, X)})
            if k != old['majority_k']:
                print(f"  {method.upper():10s}  k {old['majority_k']} -> {k} with shared structural votes "
                      f"(its 30 runs will be redone)")
            majority_k_results[dataset_name][method] = {'majority_k': k, 'votes': votes}
            ckpt['ksel'][(dataset_name, method)] = majority_k_results[dataset_name][method]
            save_ckpt(ckpt)
            print(f"  {method.upper():10s}  Votes={[votes[c] for c in VOTE_ORDER]} → k={k}  [restored from checkpoint]")
            continue
        t0 = time.perf_counter()
        k, votes = majority_vote_k_for_method(X, sen, method, dataset_name)
        elapsed  = time.perf_counter() - t0
        majority_k_results[dataset_name][method] = {'majority_k': k, 'votes': votes}
        ckpt['ksel'][(dataset_name, method)] = majority_k_results[dataset_name][method]   # NEW
        save_ckpt(ckpt)                                                                    # NEW
        print(f"  {method.upper():10s}  Votes={list(votes.values())[:6]} → k={k}  [{elapsed:.1f}s]")

# Summary
print("\n=== PER-ALGORITHM MAJORITY VOTE K SUMMARY ===")
for ds in dataset_names:
    ks = {m: majority_k_results[ds][m]['majority_k'] for m in methods}
    print(f"  {ds:25s}: {ks}")

import pandas as pd
rows = []
for ds in dataset_names:
    for m in methods:
        v = majority_k_results[ds][m]
        rows.append({'Dataset':ds,'Method':m,'Majority_k':v['majority_k'],**v['votes']})
pd.DataFrame(rows).to_csv('S2B_majority_vote_k_per_algo_kmf.csv', index=False)
print("\n✓ Saved: S2B_majority_vote_k_per_algo_kmf.csv")


Structural votes taken from Strategy 1 for 18 dataset(s)

STRATEGY 2 — PER-ALGORITHM MAJORITY VOTE K SELECTION
(silhouette/calinski/davies computed using each algorithm's own labels)

DIABETES_GENDER:
  KMEANS      Votes=[4, 2, 3, 2, 9, 4] → k=4  [restored from checkpoint]
  FAIRLET     Votes=[4, 4, 2, 4, 9, 4] → k=4  [restored from checkpoint]
  BFKM        Votes=[4, 2, 5, 2, 9, 4] → k=4  [restored from checkpoint]

DIABETES_RACE:
  KMEANS      Votes=[4, 2, 3, 7, 10, 6] → k=4  [restored from checkpoint]
  FAIRLET     Votes=[4, 5, 2, 10, 10, 6] → k=10  [restored from checkpoint]
  BFKM        Votes=[4, 2, 5, 2, 10, 6] → k=2  [restored from checkpoint]

DIABETES_COMBINED:
  KMEANS      Votes=[4, 2, 3, 7, 10, 4] → k=4  [restored from checkpoint]
  FAIRLET     Votes=[4, 2, 2, 7, 10, 4] → k=4  [restored from checkpoint]
  BFKM        Votes=[4, 2, 5, 2, 10, 4] → k=4  [restored from checkpoint]

DUTCH_GENDER:
  KMEANS      Votes=[10, 2, 9, 10, 10, 10] → k=10  [restored from checkpoint]
  FAI

In [20]:
## Cell 9: Run Experiments with Majority Vote K

In [21]:
import time

SEEDS        = list(range(30))


# ORIGINAL (Reviewer 3, Comment 2, 4 & 6 -- kept for reference, not executed):
# UTIL_COLS = ['Inertia','Silhouette','Calinski_Harabasz','Davies_Bouldin','BCSS','Variance_Ratio']
# FAIR_COLS = ['Balance','SPD','Disparate_Impact','Entropy']
#
# def run_single_s2(X, sensitive, k, method, seed):
#     np.random.seed(seed)
#     if method == 'kmeans':
#         labels, _ = standard_kmeans(X, k, random_state=seed)
#     elif method == 'fairlet':
#         labels, _ = fairlet_clustering(X, sensitive, k)
#     elif method == 'bfkm':
#         labels, _ = bfkm_clustering(X, sensitive, k)
#     elif method == 'fair_centroid':
#         labels, _ = fair_centroid_clustering(X, sensitive, k, max_iter=10, random_state=seed)
#     elif method == 'postprocessing_nfp':
#         labels, _ = postprocessing_clustering_nfp(X, sensitive, k, max_iter=15, random_state=seed)
#     elif method == 'postprocessing_gini':
#         labels, _ = postprocessing_clustering_gini(X, sensitive, k, max_iter=15, random_state=seed)
#     elif method == 'rawlsian':
#         labels, _ = rawlsian_kmeans(X, sensitive, k, n_runs=10, random_state=seed)
#     quality  = calculate_quality_metrics(X, labels)
#     fairness = {
#         'Balance':          calculate_balance(labels, sensitive),
#         'SPD':              calculate_spd(labels, sensitive),
#         'Disparate_Impact': calculate_disparate_impact(labels, sensitive),
#         'Entropy':          calculate_entropy_metric(labels, sensitive)
#     }
#     return quality, fairness

# FIXED (Reviewer 3, Comment 6): Calinski_Harabasz and BCSS removed --
# algebraically redundant with Variance_Ratio and Inertia respectively
# under fixed k (see Strategy1_PartA_30runs.ipynb for the derivation).
UTIL_COLS = ['Inertia','Silhouette','Davies_Bouldin','Variance_Ratio']
# FIXED (Reviewer 3, Comment 2): 'DI_dist' = |DI-1| added as a separate
# ranking-oriented field (ideal = 0, lower always better) alongside the
# raw 'Disparate_Impact' (ideal = 1).
FAIR_COLS = ['Balance','SPD','Disparate_Impact','DI_dist','Entropy']

def run_single_s2(X, sensitive, k, method, seed):
    np.random.seed(seed)
    if method == 'kmeans':
        labels, _ = standard_kmeans(X, k, random_state=seed)
    elif method == 'fairlet':
        labels, _ = fairlet_clustering(X, sensitive, k)
    elif method == 'bfkm':
        labels, _ = bfkm_clustering(X, sensitive, k)
    elif method == 'fair_centroid':
        labels, _ = fair_centroid_clustering(X, sensitive, k, max_iter=10, random_state=seed)
    elif method == 'postprocessing_nfp':
        # FIXED (Reviewer 3, Comment 4): max_iter 15 -> 30, matching Table 2 (unused in this file's `methods` list, kept for consistency)
        labels, _ = postprocessing_clustering_nfp(X, sensitive, k, max_iter=30, random_state=seed)
    elif method == 'postprocessing_gini':
        # FIXED (Reviewer 3, Comment 4): max_iter 15 -> 30, matching Table 2 (unused in this file's `methods` list, kept for consistency)
        labels, _ = postprocessing_clustering_gini(X, sensitive, k, max_iter=30, random_state=seed)
    elif method == 'rawlsian':
        # FIXED (Reviewer 3, Comment 4): n_runs 10 -> 30, matching Table 2 (unused in this file's `methods` list, kept for consistency)
        labels, _ = rawlsian_kmeans(X, sensitive, k, n_runs=30, random_state=seed)
    quality  = calculate_quality_metrics(X, labels)
    di_raw = calculate_disparate_impact(labels, sensitive)
    fairness = {
        'Balance':          calculate_balance(labels, sensitive),
        'SPD':              calculate_spd(labels, sensitive),
        'Disparate_Impact': di_raw,                 # FIXED: raw DI, ideal = 1 (Comment 2)
        'DI_dist':          abs(di_raw - 1.0),       # FIXED: new field, |DI-1|, ideal = 0, used for ranking (Comment 2)
        'Entropy':          calculate_entropy_metric(labels, sensitive)
    }
    return quality, fairness

methods   = ['kmeans', 'fairlet', 'bfkm']
final_results_30 = {}

print("\n" + "="*80)
print("RUNNING 30-SEED EXPERIMENTS — Strategy 2 PartB KMF")
print("="*80)

for dataset_name in dataset_names:
    X         = datasets[dataset_name]['X']
    sensitive = datasets[dataset_name]['sensitive']
    final_results_30[dataset_name] = {}

    print(f"\n{dataset_name.upper()}:")

    for method in methods:
        # FIX: k is now looked up per-method, inside the method loop,
        # instead of once outside the loop using a stale leftover `method` value.
        k      = majority_k_results[dataset_name][method]['majority_k']
        seeds  = SEEDS
        n_runs = len(seeds)

        # ── NEW: checkpoint / resume ──
        key = (dataset_name, method)
        if key in ckpt['done'] and ckpt['done'][key]['k'] == k:
            final_results_30[dataset_name][method] = ckpt['done'][key]
            _r = ckpt['done'][key]
            print(f"  {method.upper():25s} (k={k})  [restored from checkpoint, skipped]"
                  f"  Sil={_r['mean_quality']['Silhouette']:.3f}"
                  f"  Bal={_r['mean_fairness']['Balance']:.3f}")
            continue
        part = ckpt['partial'].get(key)
        if part is None or part['k'] != k:
            part = {'k': k, 'seeds_done': [], 'runs_q': [], 'runs_f': [], 'elapsed': 0.0}
        elif part['seeds_done']:
            print(f"  {method.upper():25s} (k={k})  resuming from checkpoint: {len(part['seeds_done'])}/{n_runs} seeds already done")
        runs_q, runs_f = part['runs_q'], part['runs_f']

        for seed in seeds:
            if seed in part['seeds_done']:
                continue
            _t = time.perf_counter()
            q, f = run_single_s2(X, sensitive, k, method, seed)
            runs_q.append(q); runs_f.append(f)
            part['seeds_done'].append(seed)
            part['elapsed'] += time.perf_counter() - _t     # count compute time only
            ckpt['partial'][key] = part
            save_ckpt(ckpt)                                   # save after every seed
        elapsed = part['elapsed']

        def agg(runs, cols):
            return (
                {c: float(np.mean([r[c] for r in runs])) for c in cols if c in runs[0]},
                {c: float(np.std( [r[c] for r in runs])) for c in cols if c in runs[0]}
            )
        mq, sq = agg(runs_q, UTIL_COLS)
        mf, sf = agg(runs_f, FAIR_COLS)

        final_results_30[dataset_name][method] = {
            'k': k, 'n_runs': n_runs,
            'runtime_sec': elapsed,   # FIX: store the actually-measured runtime for this run
            'mean_quality': mq, 'std_quality': sq,
            'mean_fairness': mf, 'std_fairness': sf,
            'all_quality': runs_q, 'all_fairness': runs_f
        }
        # NEW: all 30 seeds of this dataset x method finished -> move to 'done', drop the partial entry
        ckpt['done'][key] = final_results_30[dataset_name][method]
        ckpt['partial'].pop(key, None)
        save_ckpt(ckpt)
        print(f"  [{n_runs} runs] {method.upper():25s} (k={k})"
              f"  Sil={mq['Silhouette']:.3f}±{sq['Silhouette']:.3f}"
              f"  Bal={mf['Balance']:.3f}±{sf['Balance']:.3f}"
              f"  [{elapsed:.1f}s]")

print("\n✓ ALL 30-SEED EXPERIMENTS COMPLETED")



RUNNING 30-SEED EXPERIMENTS — Strategy 2 PartB KMF

DIABETES_GENDER:
  KMEANS                    (k=4)  [restored from checkpoint, skipped]  Sil=0.003  Bal=0.377
  FAIRLET                   (k=4)  resuming from checkpoint: 4/30 seeds already done
  [30 runs] FAIRLET                   (k=4)  Sil=0.056±0.198  Bal=0.755±0.033  [10842.8s]
  [30 runs] BFKM                      (k=4)  Sil=-0.109±0.098  Bal=0.546±0.202  [8539.1s]

DIABETES_RACE:
  [30 runs] KMEANS                    (k=4)  Sil=0.014±0.050  Bal=0.115±0.098  [6822.4s]
  [30 runs] FAIRLET                   (k=10)  Sil=-0.004±0.006  Bal=0.250±0.000  [7647.8s]
  [30 runs] BFKM                      (k=2)  Sil=0.080±0.129  Bal=0.213±0.053  [7097.8s]

DIABETES_COMBINED:
  [30 runs] KMEANS                    (k=4)  Sil=0.014±0.050  Bal=0.048±0.076  [7234.3s]
  [30 runs] FAIRLET                   (k=4)  Sil=0.031±0.071  Bal=0.182±0.006  [7696.2s]
  [30 runs] BFKM                      (k=4)  Sil=-0.100±0.106  Bal=0.096±0.058  [7796.5s]

In [22]:
## Cell 10: Create Comparison Tables

## Save Results

In [24]:
import pandas as pd

# NOTE: runtime_sec is now taken directly from final_results_30[ds][method]['runtime_sec'],
# which was measured live in Cell 9 (time.perf_counter() around the 30-seed loop).
# The old hardcoded runtime_data dict (copy-pasted from a previous run's printed output) has been removed.

# Save pkl
with open('S2B_results_kmf_30runs-1.pkl', 'wb') as f:
    pickle.dump({
        'majority_k_results': majority_k_results,
        'final_results_30':   final_results_30,
        'datasets':           dataset_names,
        'seeds':              SEEDS,
    }, f)
print("✓ Saved: S2B_results_kmf_30runs-1.pkl")

# Runtime table
rt_rows = []
for ds in final_results_30:
    for m in final_results_30[ds]:
        r = final_results_30[ds][m]
        rt_rows.append({'Dataset': ds, 'Method': m.upper(),
                        'n_runs':      r['n_runs'],
                        'runtime_sec': round(r['runtime_sec'], 2) if r.get('runtime_sec') else None})
rt_df = pd.DataFrame(rt_rows)
print("\n=== RUNTIME SUMMARY (seconds) ===")
print(rt_df.pivot(index='Dataset', columns='Method', values='runtime_sec').round(1).to_string())
rt_df.to_csv('S2B_results_kmf_30runs-1_runtime.csv', index=False)

# Summary CSV
# FIXED: summary CSV now lists the same metrics as the pkl (DI_dist included; CH/BCSS removed)
UTIL_COLS = ['Inertia','Silhouette','Davies_Bouldin','Variance_Ratio']
FAIR_COLS = ['Balance','SPD','Disparate_Impact','DI_dist','Entropy']
rows = []
for ds in final_results_30:
    for method in final_results_30[ds]:
        r = final_results_30[ds][method]
        row = {'Dataset': ds, 'Method': method, 'k': r['k'], 'N_runs': r['n_runs'],
               'runtime_sec': round(r['runtime_sec'], 2) if r.get('runtime_sec') else None}
        for c in UTIL_COLS:
            if c in r['mean_quality']:
                row[f'{c}_mean'] = round(r['mean_quality'][c], 4)
                row[f'{c}_std']  = round(r['std_quality'][c],  4)
        for c in FAIR_COLS:
            if c in r['mean_fairness']:
                row[f'{c}_mean'] = round(r['mean_fairness'][c], 4)
                row[f'{c}_std']  = round(r['std_fairness'][c],  4)
        rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_csv('S2B_results_kmf_30runs-1_summary.csv', index=False)
print("✓ Saved: S2B_results_kmf_30runs-1_summary.csv")
print("\nPreview:")
print(summary_df[['Dataset','Method','k','N_runs','runtime_sec',
                   'Silhouette_mean','Silhouette_std',
                   'Balance_mean','Balance_std']].head(10).to_string(index=False))


✓ Saved: S2B_results_kmf_30runs-1.pkl

=== RUNTIME SUMMARY (seconds) ===
Method               BFKM  FAIRLET   KMEANS
Dataset                                    
diabetes_combined  7796.5   7696.2   7234.3
diabetes_gender    8539.1  10842.8  13197.2
diabetes_race      7097.8   7647.8   6822.4
dutch_gender       1256.1   2245.7   1335.3
meps_combined       452.0    586.0    455.1
meps_gender         438.8    762.7    452.1
meps_race           448.5    690.4    431.6
✓ Saved: S2B_results_kmf_30runs-1_summary.csv

Preview:
          Dataset  Method  k  N_runs  runtime_sec  Silhouette_mean  Silhouette_std  Balance_mean  Balance_std
  diabetes_gender  kmeans  4      30     13197.16           0.0030          0.0387        0.3768       0.3300
  diabetes_gender fairlet  4      30     10842.81           0.0559          0.1976        0.7547       0.0330
  diabetes_gender    bfkm  4      30      8539.13          -0.1090          0.0981        0.5458       0.2019
    diabetes_race  kmeans  4      3